# 12. Paoay: multi-year land activity (cropland mask)

Map where land is actively cropped, using 2019–2026 half-month Sentinel-2 composites, and compare that with the frozen January–April 2024 crop map from notebook 10. The method is ported from the archived Atok cropping-pattern notebook ([archive/14](../archive/14_atok_unsupervised.ipynb)): growth cycles per pixel, K-means on eight pixel summaries, and fixed naming rules.

- **Descriptive and unvalidated.** Land-activity classes are rule-named clusters, not crop/noncrop ground truth.
- **Crop predictions are unchanged.** The crop map is only split by land activity; nothing is masked, refitted or relabelled.
- **Independent check:** NAMRIA ELUM 2023 polygons (forest, built-up, pasture) test the non-crop side. The KML reference areas, which are all farmed, test the crop side.


## 1. Setup

The land-use map is `atok_landuse_path`.


In [ ]:
from pathlib import Path
import json
import sys
repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import land_activity
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress
settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
imagery_root = local_path(local_settings.get("atok_imagery_root", "./data/imagery/paoay_sentinel2_csplus"))
landuse_path = local_path(local_settings.get("atok_landuse_path", "./data/atok/elum2023_benguet.gpkg"))
rules = dict(land_activity.RULES)
show_preview = True
crop_pointer = paoay_root / "crop_baseline/crop_baseline_stage.json"
activity_root = paoay_root / "land_activity"
activity_pointer = activity_root / "land_activity_stage.json"
write_stage_pointer(activity_pointer, status="pending", reason="Notebook 12 land activity is incomplete")
stages = ["Verify inputs", "Classify land activity", "Check against references and NAMRIA", "Split the crop map", "Map and publish", "Other months"]
notebook_progress = pipeline_progress.start_notebook("12_paoay_land_activity", stages, report_folder.parent / "progress", {"rules": rules})


## 2. Verify inputs

Uses the current crop baseline and every composite present for the configured years; each file is checksummed so a changed composite selects a new bundle. A previous bundle with the same inputs, rules and code is reused.


In [ ]:
pipeline_progress.begin_stage("Verify inputs")
crop_handoff = resolve_stage_pointer(crop_pointer)
request = land_activity.prepare_request(crop_handoff, imagery_root, landuse_path, activity_root, rules)
composites = request["identity"]["composites"]
years = {}
for item in composites:
    years[item["file"].split("/")[0]] = years.get(item["file"].split("/")[0], 0) + 1
show_table([{"year": year, "half-month composites": count} for year, count in sorted(years.items())], title=f"{len(composites)} composites")
show_table([{"rule": key.replace("_", " "), "value": ", ".join(map(str, value)) if isinstance(value, list) else value} for key, value in rules.items()], title="Land-activity rules (archived notebook 14)")
pipeline_progress.end_stage()


## 3. Classify land activity

Pixels need at least 60 clear half-months. Gaps of up to two half-months are interpolated and the series is smoothed with a 3-step running median. A cycle is an NDVI peak with prominence ≥ 0.15, bounded by the lowest points towards neighbouring peaks (at most 6 months away), lasting 2–8 months with ≥ 60% clear half-months.

Eight summaries per pixel (NDVI 10th/50th/90th percentiles, cycles per year, mean cycle amplitude, share of half-months with NDVI < 0.3, median NDMI, median B11) are standardized and clustered with K-means. K from 2 to 8 is chosen by silhouette among stable solutions (seed and 80%-subsample ARI ≥ 0.8). Clusters are named from their medians:

| Name | Rule |
|---|---|
| bare / built | NDVI 90th percentile < 0.35 |
| perennial vegetation | NDVI 10th percentile ≥ 0.5 and < 1 cycle per year |
| active cropland | ≥ 1 cycle per year and mean cycle amplitude ≥ 0.2 |
| mixed / low activity | everything else |


In [ ]:
pipeline_progress.begin_stage("Classify land activity")
activity_handoff = land_activity.build(request)
bundle = land_activity.load_bundle(activity_handoff)
show_table(bundle["tables"]["k_selection"], title="K selection")
show_table(bundle["tables"]["cluster_centers"], title="Cluster medians and names")
pipeline_progress.end_stage()


## 4. Check against references and NAMRIA

**Crop side:** share of each KML reference area (shrunk 10 m inward) mapped as active cropland. These areas are farmed, so most of their pixels should be active cropland; rotations, fallow years and seedling beds lower it.

**Non-crop side:** NAMRIA ELUM 2023 polygons, shrunk 5 m inward. ELUM was drawn from high-resolution imagery, LGU maps and a 2023 field validation, independent of these features. Its `crops` class includes fallow terraces, so that row is a rough comparison only. Shares exclude pixels with insufficient clear observations.


In [ ]:
pipeline_progress.begin_stage("Check against references and NAMRIA")
show_table(bundle["tables"]["reference_area_check"], title="KML reference areas by land activity")
show_table(bundle["tables"]["namria_check"], title="NAMRIA ELUM 2023 types by land activity")
headline = land_activity.headlines(bundle)
print(f"Reference-area pixels mapped as active cropland: {headline['reference_area_pixels_in_active_cropland']:.0%}")
print(f"NAMRIA forest/built-up/pasture pixels left out of active cropland: {headline['namria_noncrop_pixels_outside_active_cropland']:.0%}")
print(f"NAMRIA crops pixels mapped as active cropland (rough; includes fallow): {headline['namria_crops_pixels_in_active_cropland']:.0%}")
pipeline_progress.end_stage()


## 5. Split the crop map by land activity

The frozen January–April 2024 crop baseline (reference-polygon union) is cross-tabulated with land activity. A carrot/cabbage/potato prediction outside active cropland is a candidate false positive to review; Unknown inside active cropland is cropland where the three-crop model was not confident (possibly another crop, a fallow phase or a crop outside its training window). Nothing is removed from the crop map.


In [ ]:
pipeline_progress.begin_stage("Split the crop map")
rows = [row for row in bundle["tables"]["crop_map_by_land_activity"] if int(row["pixels"])]
show_table(rows, title="Crop baseline categories by land activity")
print(f"Carrot/cabbage/potato predictions inside active cropland: {headline['crop_predictions_in_active_cropland']:.0%}")
print(f"Unknown crop type inside active cropland: {headline['unknown_in_active_cropland']:.0%}")
print("Saved tables:", bundle["folder"])
pipeline_progress.end_stage()


## 6. Map and publish

Left: land activity 2019–2026. Middle: the January–April 2024 crop map as saved. Right: the same crop map shown only where the land-activity map says *active cropland*; pixels whose 2019–2026 history looks like perennial vegetation, mixed / low activity or bare / built are hidden in blue-grey. The GeoTIFFs `land_activity.tif` (class codes) and `pixel_summaries.tif` (the eight summaries) are in the bundle folder.


In [ ]:
pipeline_progress.begin_stage("Map and publish")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    figure = land_activity.plot(bundle)
    preview = activity_root / "previews" / (bundle["manifest"]["bundle_id"] + "_land_activity.png")
    preview.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Preview:", preview)
land_activity.load_bundle(activity_handoff)
write_stage_pointer(activity_pointer, activity_handoff)
print("Land-activity handoff:", activity_handoff)
print("Crop predictions changed: no")
pipeline_progress.end_stage()


## 7. Other months for future research

In [ ]:
pipeline_progress.begin_stage("Other months")
show_table([{**row, **{key: f"{float(row[key]):.0%}" for key in ("clear_share_active_cropland", "clear_share_study_area", "cycle_start_share", "cycle_peak_share")}}
            for row in bundle["tables"]["monthly_activity"]], title="Month by month, active cropland 2019–2026")
window_rows = land_activity.window_summary(bundle["tables"]["four_month_windows"])
show_table([{**row, "median_complete_share": f"{row['median_complete_share']:.0%}", "best_complete_share": f"{row['best_complete_share']:.0%}"}
            for row in window_rows], title="Four-month windows: active-cropland pixels with all eight half-months clear")
ranked = sorted((row for row in window_rows if row["years_with_all_composites"]), key=lambda row: -row["median_complete_share"])
print("Windows ranked by median complete share:", ", ".join(f"{row['window']} {row['median_complete_share']:.0%}" for row in ranked))
if show_preview:
    figure = land_activity.plot_months(bundle)
    preview = activity_root / "previews" / (bundle["manifest"]["bundle_id"] + "_months.png")
    figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Preview:", preview)
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
